# Forecast Product Demand with 3 Lines of Code

**Use Case:** E-commerce demand forecasting for inventory optimization
**Dataset:** 6 months of synthetic daily sales with weekly seasonality
**API Plan:** Free tier (works with 500 credits/month)

This notebook shows how to use TSFA to predict the next 14 days of retail demand and compute a stock planning range from the 95% confidence interval.


In [ ]:
# Install dependencies (if needed)
# !pip install requests matplotlib

import requests
import matplotlib.pyplot as plt
import random

BASE_URL = "http://localhost:8000/v1"
HEADERS = {"Content-Type": "application/json", "X-Plan": "free"}


## Step 1 — Generate Realistic Sales Data

In production, replace this with: `pd.read_csv('your_sales_data.csv')['sales'].tolist()`

In [ ]:
def generate_retail_series(n=180, seed=42):
    """6 months of daily sales: trend + weekly seasonality + noise."""
    rng = random.Random(seed)
    base = 100.0
    values = []
    for i in range(n):
        trend = base * (1 + 0.005 / 7) ** i        # +0.5%/week growth
        weekday = i % 7
        seasonal = 1.35 if weekday in (4, 5) else (0.85 if weekday == 6 else 1.0)
        noise = rng.gauss(0, base * 0.04)
        values.append(round(max(0, trend * seasonal + noise), 1))
    return values

series = generate_retail_series()
print(f"Series: {len(series)} days of data")
print(f"Range: {min(series):.0f} – {max(series):.0f} units/day")
print(f"Last 7 days: {series[-7:]}")


## Step 2 — Call TSFA API (3 lines of code)

In [ ]:
# ── 3 lines of code ──────────────────────────────────────────────
response = requests.post(f"{BASE_URL}/forecast/univariate", headers=HEADERS,
    json={"series": series, "horizon": 14, "frequency": "D", "model": "auto",
          "confidence_levels": [0.8, 0.95]})
data = response.json()
# ─────────────────────────────────────────────────────────────────

print(f"Model used   : {data['model_used']}")
print(f"Inference    : {data['meta']['inference_time_ms']}ms")
print(f"Trend        : {data['diagnostics']['trend']}")
print(f"Forecast mean (first 7 days): {[round(v,1) for v in data['forecast']['mean'][:7]]}")


## Step 3 — Visualize Historical Data + Forecast

In [ ]:
mean  = data["forecast"]["mean"]
l80   = data["forecast"]["lower_80"]
u80   = data["forecast"]["upper_80"]
l95   = data["forecast"]["lower_95"]
u95   = data["forecast"]["upper_95"]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("white")

hist_x  = list(range(len(series)))
fore_x  = list(range(len(series) - 1, len(series) + len(mean)))
fore_mean = [series[-1]] + mean
fore_l80 = [series[-1]] + l80;  fore_u80 = [series[-1]] + u80
fore_l95 = [series[-1]] + l95;  fore_u95 = [series[-1]] + u95

ax.fill_between(fore_x, fore_l95, fore_u95, alpha=0.15, color="gray", label="95% CI")
ax.fill_between(fore_x, fore_l80, fore_u80, alpha=0.25, color="gray", label="80% CI")
ax.plot(hist_x, series, color="#2196F3", linewidth=1.5, label="Historical")
ax.plot(fore_x, fore_mean, color="#FF9800", linewidth=2.5, linestyle="--", label="Forecast (14 days)")

ax.set_title("Retail Demand Forecast — 14-Day Ahead", fontsize=13, fontweight="bold")
ax.set_xlabel("Day"); ax.set_ylabel("Units sold")
ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig("outputs/01_retail_forecast.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()


## Step 4 — Business Insight: Stock Planning Range

In [ ]:
lower_plan = round(min(l95))
upper_plan = round(max(u95))
peak_day   = mean.index(max(mean)) + 1
avg_demand = round(sum(mean) / len(mean), 1)

print("=" * 55)
print("Stock Planning Recommendation (next 14 days)")
print("=" * 55)
print(f"Expected average demand : {avg_demand} units/day")
print(f"Peak demand day         : Day {peak_day} ({round(max(mean), 1)} units)")
print()
print(f"With 95% confidence interval:")
print(f"  Plan stock between {lower_plan} and {upper_plan} units total")
print(f"  (covering the full 14-day forecast horizon)")
print()
print(f"Recommended safety stock: {upper_plan - round(sum(mean))} units")
